In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

INPUT_PATH  = "../../../data/phase2/validation_results.parquet"
OUTPUT_PATH = "../../../data/phase2/metrics_summary.parquet"

NADEX_WIN_PAYOUT  = 0.80
NADEX_LOSS_COST   = 1.00

In [ ]:
def win_rate(labels: pd.Series) -> float:
    """Fraction of trades that expired ITM."""
    return float(labels.mean())


def brier_score(labels: pd.Series, probs: pd.Series) -> float:
    """Mean squared error between predicted probability and binary outcome."""
    return float(((probs - labels) ** 2).mean())


def expected_value(wr: float, win_payout: float = NADEX_WIN_PAYOUT,
                   loss_cost: float = NADEX_LOSS_COST) -> float:
    """EV per $1 risked: wr * win_payout - (1 - wr) * loss_cost."""
    return wr * win_payout - (1 - wr) * loss_cost


def summarise_metrics(
    results: pd.DataFrame,
    label_col: str = "label",
    prob_col: str = "prob_itm",
) -> dict:
    """
    Compute all three metrics for both rules-only and model selections.

    Rules-only baseline:
      - Uses rows where signal_valid_enc == 1
      - Assigns constant probability = overall win rate (naive baseline)

    Model selection:
      - Uses all rows, sorted by prob_itm descending
      - Selects top 50% by probability as "take this trade"

    Returns dict with keys:
      rules_win_rate, rules_brier, rules_ev,
      model_win_rate, model_brier, model_ev,
      n_rules_trades, n_model_trades
    """
    rules_rows = results[results["signal_valid_enc"] == 1].copy()
    overall_wr = win_rate(results[label_col])
    rules_rows["const_prob"] = overall_wr

    model_threshold = results[prob_col].quantile(0.50)
    model_rows = results[results[prob_col] >= model_threshold].copy()

    return {
        "rules_win_rate":   win_rate(rules_rows[label_col]),
        "rules_brier":      brier_score(rules_rows[label_col], rules_rows["const_prob"]),
        "rules_ev":         expected_value(win_rate(rules_rows[label_col])),
        "n_rules_trades":   len(rules_rows),
        "model_win_rate":   win_rate(model_rows[label_col]),
        "model_brier":      brier_score(model_rows[label_col], model_rows[prob_col]),
        "model_ev":         expected_value(win_rate(model_rows[label_col])),
        "n_model_trades":   len(model_rows),
    }


def plot_calibration(results: pd.DataFrame, label_col: str = "label",
                     prob_col: str = "prob_itm") -> None:
    """
    Plot calibration curve: fraction of positives vs mean predicted probability.
    A perfectly calibrated model lies on the diagonal.
    """
    fraction_of_positives, mean_predicted = calibration_curve(
        results[label_col], results[prob_col], n_bins=10, strategy="uniform"
    )
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax.plot(mean_predicted, fraction_of_positives, "s-", label="Logistic regression")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives (actual win rate)")
    ax.set_title("Calibration Curve — Logistic Regression")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
results = pd.read_parquet(INPUT_PATH)
metrics = summarise_metrics(results)

print("=" * 50)
print("RULES-ONLY BASELINE")
print(f"  Trades selected:  {metrics['n_rules_trades']}")
print(f"  Win rate:         {metrics['rules_win_rate']:.3f}")
print(f"  Brier score:      {metrics['rules_brier']:.4f}")
print(f"  Expected value:   {metrics['rules_ev']:+.4f} per $1 risked")

print("\nMODEL (top-50% by prob_itm)")
print(f"  Trades selected:  {metrics['n_model_trades']}")
print(f"  Win rate:         {metrics['model_win_rate']:.3f}")
print(f"  Brier score:      {metrics['model_brier']:.4f}")
print(f"  Expected value:   {metrics['model_ev']:+.4f} per $1 risked")
print("=" * 50)

In [ ]:
plot_calibration(results)

In [ ]:
fold_metrics = []
for fold_idx in sorted(results["fold"].unique()):
    fold = results[results["fold"] == fold_idx]
    m = summarise_metrics(fold)
    m["fold"] = fold_idx
    fold_metrics.append(m)

fold_df = pd.DataFrame(fold_metrics).set_index("fold")
print(fold_df[["rules_win_rate", "model_win_rate", "rules_ev", "model_ev",
               "rules_brier", "model_brier"]].round(4).to_string())

In [ ]:
summary_df = pd.DataFrame([metrics])
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
summary_df.to_parquet(OUTPUT_PATH, index=False)
print(f"\nSaved metrics summary to {OUTPUT_PATH}")